In [78]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
import pandas as pd
import time
import io
import re
import json
import math

#chrome.exe --remote-debugging-port=9222 --user-data-dir="D:\Antony\chrome-profile"

In [ ]:
class ScreenerFundamentalExtractor:

    def __init__(self):
        print("Starting Chrome with remote debugging...")
        chrome_options = Options()
        chrome_options.debugger_address = "127.0.0.1:9222"
        self.driver = webdriver.Chrome(options=chrome_options)
        print("Connected to Chrome.")


    def open_company(self, symbol):

        url = f"https://www.screener.in/company/{symbol}/"

        self.driver.get(url)

        time.sleep(3)


    def get_html(self):

        return self.driver.page_source


    def parse_ratios(self, soup):

        ratios = {}

        ul = soup.select_one("#top-ratios")

        if not ul:
            return ratios

        items = ul.find_all("li")

        for item in items:

            name = item.find("span", class_="name")
            value = item.find("span", class_="number")

            if name and value:

                ratios[name.text.strip()] = value.text.strip()

        return ratios


    def extractTableGlobalUnit(self, section):
        if not section:
            return None

        elem = section.select_one("div.flex-row.flex-space-between.flex-gap-16 > div:nth-child(1) > p")
        if not elem:
            return None

        text = elem.get_text(strip=True)
        match = re.search(r'(Rs\.|USD)\s*(Crores|Lakhs|Millions)', text)
        unit = match.group() if match else None

        return unit

    def parse_table(self, soup, section_id):

        section = soup.find("section", id=section_id)
        table = section.find("table", class_="data-table")
        if not table:
            return None

        df = pd.read_html(io.StringIO(str(table)))[0]
        df.iloc[:,0] = (
            df.iloc[:,0]
            .str.replace("\xa0", " ", regex=False)
            .str.replace("+", "", regex=False)
            .str.strip()
        )
        df["unit"] = self.extractTableGlobalUnit(section)
        return df
    

    def extractUnitFromData(self, metricsNames):
        # dict mapping
        unit_map = {
            #quaterly results, Profit & Loss
            "OPM": "%",
            "TAX": "%",
            "Tax": "%",
            "EPS": "Rs",
            "Dividend Payout": "%",
            
            #Ratios
            "Debtor Days": "Days",
            "Inventory Days": "Days",
            "Days Payable": "Days",
            "Cash Conversion Cycle": "Days",
            "Working Capital Days": "Days",
            "ROCE": "%",

            # insights
            "Installed Production Capacity": "Kg",
            "Actual Production Volume": "Kg",
            "Capacity Utilization": "%",
            "New Facility Land Area": "Sq. Mtrs",
            "R&D Specialist Strength": "Number",
            "Revenue Concentration - Top 10 Customers": "%",
            "Revenue Concentration - Top 5 Customers": "%",
            "Total Employee Strength": "Number",
            "Working Capital Cycle": "Days",
            "Number of Branches Number": "Number",
            "Home Loan Market Share": "%",
            "Transactions through alternate channels": "%",
            "Auto Loan Market Share": "%",
            "Registered Users on YONO": "Number in Crore",
            "CASA Market Share": "%",
            "Domestic Deposit Market Share": "%",
            

            # Shareholding Pattern
            "FIIs": "%",
            "DIIs": "%",
            "Promoters": "%",
            "Public": "%",
            "No. of Shareholders": "Number",
            "Government": "%",
        }

        # Step 1: detect unit
        unit = next((u for k,u in unit_map.items() if k in metricsNames), None)
        if unit:
            return unit
        insight_units = ["Kg", "%", "Sq. Mtrs", 
                         "Rs", "MT", "MU", "MTPA", "Meters","MMT ・Standalone data",
                         "Units", "Number", "Tonnes per Month","Rs / month","Index ・Standalone data",
                         "MMSCMD ・Standalone data", "Index", "MMSCMD",
                         "Meters Per Annum", "Million Sq. Ft.", "Million"]
        df_units = pd.read_csv("unitConfig.csv")
        escaped_units = [re.escape(u) for u in insight_units]

        pattern = r"(" + "|".join(escaped_units) + r")$"

        match = re.search(pattern, metricsNames)
        return match.group(0) if match else None


    def screener_table_to_json(self,df, table_name, reportNoneUnits=True):

        # first column is metric names
        metric_col = df.columns[0]
        

        result = {
            "table": table_name,
            "periods": list(df.columns[1:-1]),
            "metrics": {}
        }

        for _, row in df.iterrows():

            metric = row[metric_col]

            values = {}
            unit = row["unit"]
            for col in df.columns[1:-1]:
                val = row[col]

                try:
                    val_str = str(val).strip()  # e.g., "12.5%" or "1,234.56"
                    unit = self.extractUnitFromData(metric) or unit
                    val = float(str(val_str).replace('%','').replace(',',''))
                except:
                    val = None

                values[col] = val

            result["metrics"][metric] = {
                "unit": unit,
                "values": values
            }
            if reportNoneUnits and unit is None:
                print(f"❌ Error: No unit detected for metric '{metric}' in table '{table_name}'")

        return result


    def extract_metrics(self, symbol):

        self.open_company(symbol)
        html = self.get_html()
        soup = BeautifulSoup(html, "html.parser")
        #ratios = self.parse_ratios(soup)
        quarterly = self.parse_table(soup, "quarters")
        pnl = self.parse_table(soup, "profit-loss")
        balancesheet = self.parse_table(soup, "balance-sheet")
        cashflow = self.parse_table(soup, "cash-flow")
        ratios = self.parse_table(soup, "ratios")
        insights = self.parse_table(soup, "insights")
        shareholding = self.parse_table(soup, "shareholding")

        quarterly_json = self.screener_table_to_json(quarterly,"results")
        pnl_json = self.screener_table_to_json(pnl, "profit-loss")
        balancesheet_json = self.screener_table_to_json(balancesheet, "balance-sheet")
        cashflow_json = self.screener_table_to_json(cashflow, "cash-flow")
        ratios_json = self.screener_table_to_json(ratios, "ratios")
        insights_json = self.screener_table_to_json(insights, "insights")
        shareholding_json = self.screener_table_to_json(shareholding, "shareholding")

        

        stock_data = {
            "stock": symbol,
            "tables": [
                quarterly_json,
                pnl_json,
                balancesheet_json,
                cashflow_json,
                ratios_json,
                insights_json,
                shareholding_json
            ]
        }

        return stock_data

In [82]:
def replace_nan(obj):
    if isinstance(obj, dict):
        return {k: replace_nan(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan(x) for x in obj]
    elif isinstance(obj, float) and math.isnan(obj):
        return None
    else:
        return obj

In [111]:
extractor = ScreenerFundamentalExtractor()

data = extractor.extract_metrics("RELIANCE")
clean_data = replace_nan(data)
print(json.dumps(clean_data, indent=4))




Starting Chrome with remote debugging...
Connected to Chrome.
{
    "stock": "RELIANCE",
    "tables": [
        {
            "table": "results",
            "periods": [
                "Dec 2022",
                "Mar 2023",
                "Jun 2023",
                "Sep 2023",
                "Dec 2023",
                "Mar 2024",
                "Jun 2024",
                "Sep 2024",
                "Dec 2024",
                "Mar 2025",
                "Jun 2025",
                "Sep 2025",
                "Dec 2025"
            ],
            "metrics": {
                "Sales": {
                    "unit": "Rs. Crores",
                    "values": {
                        "Dec 2022": 125849.0,
                        "Mar 2023": 129674.0,
                        "Jun 2023": 122627.0,
                        "Sep 2023": 137380.0,
                        "Dec 2023": 127695.0,
                        "Mar 2024": 146832.0,
                        "Jun 2024": 129898.0,
  

In [95]:

quarterly_json = screener_table_to_json(quarterly,"results")
pnl_json = screener_table_to_json(pnl, "profit-loss")
balancesheet_json = screener_table_to_json(balancesheet, "balance-sheet")
cashflow_json = screener_table_to_json(cashflow, "cash-flow")
ratios_json = screener_table_to_json(ratios, "ratios")
insights_json = screener_table_to_json(insights, "insights")
shareholding_json = screener_table_to_json(shareholding, "shareholding")
print(shareholding_json)

{'table': 'shareholding', 'periods': ['Jun 2025', 'Sep 2025'], 'metrics': {'Promoters': {'Jun 2025': 71.54, 'Sep 2025': 71.54}, 'FIIs': {'Jun 2025': 1.46, 'Sep 2025': 0.31}, 'DIIs': {'Jun 2025': 12.81, 'Sep 2025': 6.28}, 'Public': {'Jun 2025': 14.19, 'Sep 2025': 21.88}, 'No. of Shareholders': {'Jun 2025': 2448.0, 'Sep 2025': 1141.0}}}


In [ ]:
symbol = "SACHEEROME"
stock_data = {
    "stock": symbol,
    "tables": [
        quarterly_json,
        pnl_json,
        balancesheet_json,
        cashflow_json,
        ratios_json,
        insights_json,
        shareholding_json
    ]
}
print(stock_data)

{'stock': 'SACHEEROME', 'tables': [{'table': 'results', 'periods': ['Sep 2024', 'Mar 2025', 'Sep 2025'], 'metrics': {'Sales': {'Sep 2024': 50.0, 'Mar 2025': 57.0, 'Sep 2025': 77.0}, 'Expenses': {'Sep 2024': 41.0, 'Mar 2025': 44.0, 'Sep 2025': 57.0}, 'Operating Profit': {'Sep 2024': 10.0, 'Mar 2025': 13.0, 'Sep 2025': 19.0}, 'OPM %': {'Sep 2024': 19.0, 'Mar 2025': 22.0, 'Sep 2025': 25.0}, 'Other Income': {'Sep 2024': 1.0, 'Mar 2025': 0.0, 'Sep 2025': 2.0}, 'Interest': {'Sep 2024': 0.0, 'Mar 2025': 0.0, 'Sep 2025': 0.0}, 'Depreciation': {'Sep 2024': 1.0, 'Mar 2025': 1.0, 'Sep 2025': 1.0}, 'Profit before tax': {'Sep 2024': 10.0, 'Mar 2025': 12.0, 'Sep 2025': 20.0}, 'Tax %': {'Sep 2024': 26.0, 'Mar 2025': 25.0, 'Sep 2025': 25.0}, 'Net Profit': {'Sep 2024': 7.0, 'Mar 2025': 9.0, 'Sep 2025': 15.0}, 'EPS in Rs': {'Sep 2024': 4.32, 'Mar 2025': 5.47, 'Sep 2025': 6.68}, 'Raw PDF': {'Sep 2024': nan, 'Mar 2025': nan, 'Sep 2025': nan}}}, {'table': 'profit-loss', 'periods': ['Mar 2022', 'Mar 2023', 

In [15]:
elem = soup.select_one("section#balance-sheet > div.flex-row.flex-space-between.flex-gap-16 > div:nth-child(1) > p")
if elem is not None:
    text = elem.get_text(strip=True)  # removes extra whitespace and newlines
    print(text)
    match = re.search(r'(Rs\.|USD)\s*(Crores|Lakhs|Millions)', text)
    unit = match.group() if match else None
    print(unit)

Figures in Rs. Crores
Rs. Crores
